<h1 style="text-align: center; font-family: 'Roboto', sans-serif; font-weight:bold; font-size:40px; margin: 40px 0">01 - EDA : Données Statiques</h1>

### Objectif
Ce notebook couvre l'analyse exploratoire des **3 fichiers statiques** :
- `general_data` : Données générales des employés
- `employee_survey_data` : Enquête de satisfaction des employés
- `manager_survey_data` : Enquête d'évaluation managériale

### Pourquoi séparer les fichiers d'horaires ?
Les fichiers `in_time` et `out_time` contiennent des séries temporelles (horodatages quotidiens).
Ils nécessitent une phase de feature engineering dédiée (agrégation en moyennes,
taux d'heures supplémentaires, etc...) avant de pouvoir être joints au dataset principal.
Mélanger données statiques et temporelles dans un même notebook produirait un code confus et difficile à auditer.

### Plan de ce notebook
- Importation des bibliothèques nécessaires
- Rassemblement des données
- Nettoyage des données
- Réflexion éthique

---
## **1. Importation des bibliothèques nécessaires**

In [16]:
import os

import numpy as np
import pandas as pd

# Chemins
RAW_DATA_DIR = '../data/raw'
PRE_PROCESSED_DATA_DIR = '../data/pre-processed'
CORRESPONDING_TABLE_DIR = '../data/corresponding-tables'
PROCESSED_DATA_DIR = '../data/processed'

# Clé de jointure
JOIN_KEY = 'EmployeeID'

# Colonnes à supprimer
COL_TO_DELETE = [
    'EmployeeCount',
    'Age',
    'Over18',
    'Gender',
    'MaritalStatus',
    'NumCompaniesWorked',
    'StockOptionLevel',
    'StandardHours',
    'DistanceFromHome'
]

# Mappings catégoriels
DEPARTMENT_MAP = {
    'Sales': 'Sales',
    'Research & Development': 'R&D',
    'Human Resources': 'HR'
}

EDUCATION_FIELD_MAP = {
    'Life Sciences': 'LifeSci',
    'Other': 'Other',
    'Medical': 'Medical',
    'Marketing': 'Marketing',
    'Technical Degree': 'Tech',
    'Human Resources': 'HR'
}

JOB_ROLE_MAP = {
    'Healthcare Representative': 'HealthRep',
    'Research Scientist': 'Scientist',
    'Sales Executive': 'SalesExec',
    'Human Resources': 'HR',
    'Research Director': 'ResearchDir',
    'Laboratory Technician': 'LabTech',
    'Manufacturing Director': 'MfgDir',
    'Sales Representative': 'SalesRep',
    'Manager': 'Manager'
}

# Mappings numériques
NUM_CAT_HAS_LEFT = {'No': 0, 'Yes': 1}

NUM_CAT_BUSINESS_TRAVEL = {
    'Non-Travel': 0,
    'Travel_Rarely': 1,
    'Travel_Frequently': 2
}

NUM_CAT_EDUCATION = {
    'Niveau BAC': 1,
    'BAC+2': 2,
    'BAC+3': 3,
    'BAC+5': 4,
    'Doctorat': 5
}

NUM_CAT_JOB_INVOLVEMENT = {
    'Faible': 1,
    'Moyenne': 2,
    'Importante': 3,
    'Très Importante': 4
}

NUM_CAT_PERFORMANCE_RATING = {
    'Faible': 1,
    'Bon': 2,
    'Excellent': 3,
    'Au delà des attentes': 4
}

NUM_CAT_ENVIRONMENT_SATISFACTION = {
    'Faible': 1,
    'Moyenne': 2,
    'Élevée': 3,
    'Très Élevée': 4
}

NUM_CAT_WORK_LIFE_BALANCE = {
    'Mauvais': 1,
    'Satisfaisant': 2,
    'Très Satisfaisant': 3,
    'Excellent': 4
}

# Renommage des colonnes
COL_CORR = {
    'EmployeeID': 'employee_id',
    'Education': 'education',
    'EducationField': 'education_field',
    'JobLevel': 'job_level',
    'JobRole': 'job_role',
    'Department': 'department',
    'BusinessTravel': 'business_travel',
    'MonthlyIncome': 'monthly_income',
    'PercentSalaryHike': 'percent_salary_hike',
    'TotalWorkingYears': 'total_working_years',
    'TrainingTimesLastYear': 'training_times_last_year',
    'YearsAtCompany': 'years_at_company',
    'YearsSinceLastPromotion': 'years_since_last_promotion',
    'YearsWithCurrManager': 'years_with_curr_manager',
    'EnvironmentSatisfaction': 'environment_satisfaction',
    'JobSatisfaction': 'job_satisfaction',
    'WorkLifeBalance': 'work_life_balance',
    'JobInvolvement': 'job_involvement',
    'PerformanceRating': 'performance_rating',
    'Attrition': 'has_left'
}

# Résumé de la configuration
print(f"Le répertoire de données est :\n- Raw : {os.path.abspath(RAW_DATA_DIR)}\n- Pre-processed : {os.path.abspath(PRE_PROCESSED_DATA_DIR)}\n- Processed : {os.path.abspath(PROCESSED_DATA_DIR)}\n- Table de Correspondance : {os.path.abspath(CORRESPONDING_TABLE_DIR)}")
print()
print(f"La clé de jointure pour les DataFrames est : {JOIN_KEY}")
print()
print("La configuration a été chargée.")

Le répertoire de données est :
- Raw : c:\Users\Gamas\Documents\RécupGithub\StableDiffusion\IA-groupe-02\src\data\raw
- Pre-processed : c:\Users\Gamas\Documents\RécupGithub\StableDiffusion\IA-groupe-02\src\data\pre-processed
- Processed : c:\Users\Gamas\Documents\RécupGithub\StableDiffusion\IA-groupe-02\src\data\processed
- Table de Correspondance : c:\Users\Gamas\Documents\RécupGithub\StableDiffusion\IA-groupe-02\src\data\corresponding-tables

La clé de jointure pour les DataFrames est : EmployeeID

La configuration a été chargée.


---
## **2. Rassemblement des données**

Dans cette première partie, il est question de rassembler les données dans un seul dataframe final pour pouvoir ensuite les traiter correctement. Pour ce faire, nous suivrons le plan suivant :
- Chargement des données
- Inspection de la volumétrie, des types et des valeurs manquantes
- Vérification de l'unicité de la clé de jointure
- Merge progressif des dataframes
- Visualisation du dataframe fusionné
- Bilan des valeurs manquantes post-merge

### 2.1. Chargement des données

Les fichiers d'enquête peuvent contenir la chaîne littérale `"NA"` pour signifier une non-réponse.
Par défaut, `pd.read_csv()` reconnaît `"NA"` comme valeur manquante, mais ce comportement est **implicite**.

In [17]:
# On déclare explicitement des valeurs à interpréter comme NaN
NA_VALUES = ["NA"]

df_general = pd.read_csv(os.path.join(RAW_DATA_DIR, 'general_data.csv'), na_values=NA_VALUES).convert_dtypes()
df_survey = pd.read_csv(os.path.join(RAW_DATA_DIR, 'employee_survey_data.csv'), na_values=NA_VALUES).convert_dtypes()
df_manager = pd.read_csv(os.path.join(RAW_DATA_DIR, 'manager_survey_data.csv'), na_values=NA_VALUES).convert_dtypes()

print("Les fichiers CSV ont été chargés dans des DataFrames.")

Les fichiers CSV ont été chargés dans des DataFrames.


### 2.2. Volumétrie, types et valeurs manquantes

Avant toute manipulation, on vérifie pour chaque DataFrame :
- Le nombre de lignes et colonnes (`shape`)
- Les types de chaque colonne (`dtypes`), une colonne numérique en `object` signale un problème
- Le décompte des valeurs manquantes (`isnull().sum()`)

In [18]:
datasets = {
    "General":  df_general,
    "Survey":   df_survey,
    "Manager":  df_manager,
}

for name, df in datasets.items():
    print(f"{'='*50}")
    print(f"{name} : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    print(f"{'='*50}")
    print(df.dtypes)
    print()
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("Aucune valeur manquante.")
    else:
        print(f"Valeurs manquantes :\n{missing}")
    print()

General : 4410 lignes x 24 colonnes
Age                         Int64
Attrition                  string
BusinessTravel             string
Department                 string
DistanceFromHome            Int64
Education                   Int64
EducationField             string
EmployeeCount               Int64
EmployeeID                  Int64
Gender                     string
JobLevel                    Int64
JobRole                    string
MaritalStatus              string
MonthlyIncome               Int64
NumCompaniesWorked          Int64
Over18                     string
PercentSalaryHike           Int64
StandardHours               Int64
StockOptionLevel            Int64
TotalWorkingYears           Int64
TrainingTimesLastYear       Int64
YearsAtCompany              Int64
YearsSinceLastPromotion     Int64
YearsWithCurrManager        Int64
dtype: object

Valeurs manquantes :
NumCompaniesWorked    19
TotalWorkingYears      9
dtype: int64

Survey : 4410 lignes x 4 colonnes
EmployeeID    

### 2.3. Vérification de l'unicité de la clé de jointure

Avant tout merge, il faut **prouver** que la clé est unique dans chaque DataFrame.
Si elle ne l'est pas, le merge produira un **produit cartésien** silencieux
et gonflera le nombre de lignes sans avertissement.

In [19]:
for name, df in datasets.items():
    n_duplicates = df[JOIN_KEY].duplicated().sum()
    status = "OK" if n_duplicates == 0 else f"ATTENTION : {n_duplicates} doublons !"
    print(f"{name} : clé '{JOIN_KEY}' unique : {status}")

    assert n_duplicates == 0, f"Doublons sur '{JOIN_KEY}' dans {name} — merge impossible en l'état."

General : clé 'EmployeeID' unique : OK
Survey : clé 'EmployeeID' unique : OK
Manager : clé 'EmployeeID' unique : OK


### 2.4. Merge progressif

On fusionne les 3 DataFrames **un par un** en vérifiant la volumétrie après chaque étape.
Un merge `inner` ne conserve que les lignes présentes des deux côtés :
si le nombre de lignes diminue, cela signifie qu'un employé manque dans l'un des fichiers.

In [20]:
n_expected = df_general.shape[0]

# On effectue un premier merge : General + Survey
df_merged = df_general.merge(df_survey, on=JOIN_KEY, how="inner")
print(f"Après merge General + Survey : {df_merged.shape[0]} lignes (attendu : {n_expected})")

# On effectue un second merge: Merged + Manager
df_merged = df_merged.merge(df_manager, on=JOIN_KEY, how="inner")
print(f"Après merge + Manager : {df_merged.shape[0]} lignes (attendu : {n_expected})")

# On effectue un contrôle final
assert df_merged.shape[0] == n_expected, (
    f"Le merge a perdu des lignes ! {n_expected} attendues, {df_merged.shape[0]} obtenues."
)
print(f"\nMerge validé : {df_merged.shape[0]} lignes x {df_merged.shape[1]} colonnes")

Après merge General + Survey : 4410 lignes (attendu : 4410)
Après merge + Manager : 4410 lignes (attendu : 4410)

Merge validé : 4410 lignes x 29 colonnes


### 2.5. Visualisation du Dataframe fusionné

In [21]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Age                      4410 non-null   Int64 
 1   Attrition                4410 non-null   string
 2   BusinessTravel           4410 non-null   string
 3   Department               4410 non-null   string
 4   DistanceFromHome         4410 non-null   Int64 
 5   Education                4410 non-null   Int64 
 6   EducationField           4410 non-null   string
 7   EmployeeCount            4410 non-null   Int64 
 8   EmployeeID               4410 non-null   Int64 
 9   Gender                   4410 non-null   string
 10  JobLevel                 4410 non-null   Int64 
 11  JobRole                  4410 non-null   string
 12  MaritalStatus            4410 non-null   string
 13  MonthlyIncome            4410 non-null   Int64 
 14  NumCompaniesWorked       4391 non-null   Int64 
 15

### 2.6. Bilan des valeurs manquantes post-merge

Maintenant que le DataFrame est consolidé, on dresse le bilan complet des NaN
pour savoir quelles colonnes devront être traitées avant la modélisation.

In [22]:
missing_total = df_merged.isnull().sum().sort_values(ascending=False)
missing_total = missing_total[missing_total > 0]

if missing_total.empty:
    print("Aucune valeur manquante dans le DataFrame fusionné.")
else:
    missing_pct = (missing_total / len(df_merged) * 100).round(2)
    summary = pd.DataFrame({"Nombre de NaN": missing_total, "% NaN": missing_pct})
    print(summary)

                         Nombre de NaN  % NaN
WorkLifeBalance                     38   0.86
EnvironmentSatisfaction             25   0.57
JobSatisfaction                     20   0.45
NumCompaniesWorked                  19   0.43
TotalWorkingYears                    9   0.20


---
## **3. Nettoyage des données**

Dans cette seconde partie, il est question de nettoyer les données du dataframe obtenu pour pouvoir ensuite ne garder que des colonnes propres pour notre projet. Pour ce faire, nous suivrons le plan suivant :
- Suppression des colonnes inutiles
- Remplissage des valeurs NaN
- Uniformisation
- Modification des types des colonnes
- Renommage des colonnes
- Sauvegarde du CSV

### 3.1. Suppression des colonnes inutiles

Certaines colonnes ne servent à rien pour l'analyse ultérieure des données, nous les supprimons donc.

In [23]:
# On supprime les colonnes inutiles
df_final = df_merged.drop(columns=COL_TO_DELETE)
print(f"Colonnes supprimées : {COL_TO_DELETE}")
print(f"DataFrame final : {df_final.shape[0]} lignes x {df_final.shape[1]} colonnes")
print()
print(df_final.head())

Colonnes supprimées : ['EmployeeCount', 'Age', 'Over18', 'Gender', 'MaritalStatus', 'NumCompaniesWorked', 'StockOptionLevel', 'StandardHours', 'DistanceFromHome']
DataFrame final : 4410 lignes x 20 colonnes

  Attrition     BusinessTravel              Department  Education  \
0        No      Travel_Rarely                   Sales          2   
1       Yes  Travel_Frequently  Research & Development          1   
2        No  Travel_Frequently  Research & Development          4   
3        No         Non-Travel  Research & Development          5   
4        No      Travel_Rarely  Research & Development          1   

  EducationField  EmployeeID  JobLevel                    JobRole  \
0  Life Sciences           1         1  Healthcare Representative   
1  Life Sciences           2         1         Research Scientist   
2          Other           3         4            Sales Executive   
3  Life Sciences           4         3            Human Resources   
4        Medical           5    

### 3.2. Remplissage des valeurs NaN

Au vu du nombre faible de données que nous avons, nous allons remplir les lignes NaN par la valeur médiane pour les colonnes numériques et le mode pour les valeurs catégoriques.

In [24]:
num_cols = df_final.select_dtypes(include=np.number).columns
cat_cols = df_final.select_dtypes(include=["object", "string"]).columns
print(f"Nombre de colonnes numériques : {len(num_cols)}")
print(f"Nombre de colonnes catégorielles : {len(cat_cols)}")

# On remplace les NaN des colonnes numériques par la médiane
df_final[num_cols] = df_final[num_cols].fillna(df_final[num_cols].median())
df_final[cat_cols] = df_final[cat_cols].fillna(df_final[cat_cols].mode().iloc[0])

print("Les valeurs manquantes ont été traitées !")

Nombre de colonnes numériques : 15
Nombre de colonnes catégorielles : 5
Les valeurs manquantes ont été traitées !


### 3.3. Uniformisation

On va uniformiser les noms des jobs. On commence par voir les différentes valeurs des colonnes catégoriques.

In [25]:
for col in cat_cols:
    print(f"'{col}': {str(df_final[col].unique().tolist())}")

'Attrition': ['No', 'Yes']
'BusinessTravel': ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
'Department': ['Sales', 'Research & Development', 'Human Resources']
'EducationField': ['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
'JobRole': ['Healthcare Representative', 'Research Scientist', 'Sales Executive', 'Human Resources', 'Research Director', 'Laboratory Technician', 'Manufacturing Director', 'Sales Representative', 'Manager']


On va remplacer certaines valeurs pour rendre le tout plus lisible à l'aide de tables de correspondances.

In [26]:
df_final['Department'] = df_final['Department'].map(DEPARTMENT_MAP)
df_final['EducationField'] = df_final['EducationField'].map(EDUCATION_FIELD_MAP)
df_final['JobRole'] = df_final['JobRole'].map(JOB_ROLE_MAP)

print('Les colonnes catégorielles ont été renommées pour plus de clarté.')

Les colonnes catégorielles ont été renommées pour plus de clarté.


### 3.4. Modification du type des colonnes

On va modifier certaines colonnes car elles sont plus pertinentes sous forme de nombre, par exemple :
- `Attrition`, ne possédant que les valeurs "Yes" ou "No", recevront les valeurs respectives 1 ou 0.
- `BusinessTravel` aura également sa propre matrice éditée à côté.

Puis nous allons créer les tables de correspondance pour une meilleure clarté.

In [27]:
# On modifie l'attrition pour en faire une variable binaire
df_final["Attrition"] = df_final["Attrition"].map(NUM_CAT_HAS_LEFT)
print("La variable 'Attrition' a été convertie en binaire (1 = Yes, 0 = No).")

df_final["BusinessTravel"] = df_final["BusinessTravel"].map(NUM_CAT_BUSINESS_TRAVEL)
print("La variable 'BusinessTravel' a été convertie en numérique selon le mapping défini.")

# On print les tables de correspondance
for mapping_name, mapping_dict in {
    "has_left": NUM_CAT_HAS_LEFT,
    "business_travel": NUM_CAT_BUSINESS_TRAVEL,
    "education": NUM_CAT_EDUCATION,
    "job_involvement": NUM_CAT_JOB_INVOLVEMENT,
    "performance_rating": NUM_CAT_PERFORMANCE_RATING,
    "environment_satisfaction": NUM_CAT_ENVIRONMENT_SATISFACTION,
    "work_life_balance": NUM_CAT_WORK_LIFE_BALANCE
}.items():
    pd.DataFrame([mapping_dict]).to_csv(os.path.join(CORRESPONDING_TABLE_DIR, f"{mapping_name}.csv"), index=False)

La variable 'Attrition' a été convertie en binaire (1 = Yes, 0 = No).
La variable 'BusinessTravel' a été convertie en numérique selon le mapping défini.


### 3.5. Renommage des colonnes

On renomme les colonnes du dataframe pour une meilleure clarté.

In [28]:
# On renomme les colonnes pour plus de clarté
df_final.rename(columns=COL_CORR, inplace=True)
print("Les colonnes ont été renommées pour plus de clarté.")

Les colonnes ont été renommées pour plus de clarté.


### 3.6. Sauvegarde du CSV

On sauvegarde le dataframe processed sous format CSV.

In [29]:
file_path = os.path.join(PRE_PROCESSED_DATA_DIR, "static_merged.csv")
df_final.to_csv(file_path, index=False)
print(f"Le DataFrame final a été sauvegardé dans : {os.path.abspath(file_path)}")

Le DataFrame final a été sauvegardé dans : c:\Users\Gamas\Documents\RécupGithub\StableDiffusion\IA-groupe-02\src\data\pre-processed\static_merged.csv


---
## **4. Réflexion Éthique**

Certaines des données qui nous ont été fournies par le jeu de données initial n'ont pas été retenues pour plusieurs raisons que nous allons détailler dans cette section. 

### Contradiction envers le RGPD

Il est important de rappeler que le but de notre projet est d'entraîner un modèle de prédiction à partir d'un dataset que nous allons créer. Puisque ce modèle a pour objectif du décisionnel, il ne peut contenir aucune donnée sensible qui permettrait de biaiser son jugement. Ainsi, il est interdit par le RGPD d'utiliser les données suivantes :
- Genre
- Âge
- Statut Marital

### Valeurs Redondantes ou Inutiles

Certaines valeurs permettent d'en retrouver d'autres. Puisque nous obtiendrons à la fin un dataset important, il est nécessaire de ne conserver que les données nécessaires. Aussi, par la liaison entre ces données, certains patterns peuvent être renforcés bien qu'ils ne soient pas pertinents. Ainsi, nous décidons de retirer les valeurs suivantes :
- `Over18` : Permet de déterminer si la personne a plus de 18 ans, ce qui est déterminable par l'âge et comme nous l'avons vu précédemment, il ne faut pas le prendre en compte.
- `StandardHours` : Tous les employés doivent travailler 8h par jour d'après le dataset, il est donc inutile de conserver une colonne pour ceci.
- `DistanceFromHome` : La distance entre l'employé et son lieu de travail n'a aucun influence sur son profil.

### Critères Impertinents

Enfin, certains paramètres ne sont pas liés au profil de l'individu et peuvent amener à la reconnaissance de patterns falacieux. Voici donc les valeurs à retirer selon ce contexte :
- `NumCompaniesWorked` : Le nombre d'entreprises dans lesquels un individu à travailler n'est pas déterminé par celui-ci mais est un facteur externe lié aux opportunités de travail, au réseau de la personne, etc...
- `StockOptionLevel` : L'investissement en action de l'employé est biaisé par sa richesse personnel plus que son implication dans l'entreprise.


---
<div style="display: flex; justify-content: space-between;">
  <a/>
  <a href="./02_eda_dynamic_data.ipynb">Suivant</a>
</div>